In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
path = "dbfs:/Volumes/int-cu-siampiwat/staging/report/"

In [0]:
# volume_path_202603 = "dbfs:/Volumes/int-cu-siampiwat/staging/report/report1/202603/"
volume_path_202604 = "dbfs:/Volumes/int-cu-siampiwat/staging/report/report1/202606/"

files = dbutils.fs.ls(volume_path_202604)

report1_files = [file.name for file in files if file.name.endswith('.csv/')]

display(report1_files)

In [0]:
path_mask = 'dbfs:/Volumes/int-cu-siampiwat/staging/report/report1/mask/202606/'
for file in report1_files:
    print()
    df = (spark.read
        .option("header", "true").option("inferSchema", "true")
        .csv(volume_path_202604+file)
    )

    df_marked = df.withColumn("daily_unique_visitor_cnt", F.when(F.col("daily_unique_visitor_cnt") <= 25, 25).otherwise(F.col("daily_unique_visitor_cnt")))

    save_to_csv(df_marked, path_mask+file)